# Class Exercise: Semantic Search on the Trinity College Hartford Website

In this exercise, you will build a semantic search system over the Trinity College website using a pretrained embedding model.

## What you will do
1. Crawl links from the Trinity College website.
2. Extract text from each webpage.
3. Generate embeddings for each page with a pretrained model.
4. Measure cosine similarity between documents.
5. Search the website with natural-language queries.
6. Evaluate search quality against a small ground-truth set.

## Step 0. Install packages
Run this if your environment is fresh.

In [6]:
# Uncomment if needed
#pip install -q requests beautifulsoup4 sentence-transformers pandas scikit-learn tqdm lxml

## Step 1. Import libraries

In [7]:
import re
import time
import requests
import numpy as np
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urldefrag
from collections import deque
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## Step 2. Define the seed pages and allowed domains

We will start from the Trinity home page. The official Trinity College website is under `trincoll.edu`, and the official athletics site is `bantamsports.com`.

In [8]:
SEED_URLS = [
    "https://www.trincoll.edu/",
    "https://bantamsports.com/"
]

ALLOWED_DOMAINS = {
    "www.trincoll.edu",
    "trincoll.edu",
    "bantamsports.com",
    "www.bantamsports.com",
}

## Step 3. Helper functions for crawling

In [9]:
BAD_FILE_EXTENSIONS = (
    ".pdf", ".jpg", ".jpeg", ".png", ".gif", ".svg", ".zip", ".doc", ".docx",
    ".xls", ".xlsx", ".ppt", ".pptx", ".mp4", ".mp3", ".avi", ".mov", ".webp"
)

def normalize_url(url: str) -> str:
    url, _ = urldefrag(url)
    if url.endswith("/") and len(url) > len("https://a.co/"):
        url = url.rstrip("/")
    return url

def is_allowed_url(url: str, allowed_domains=ALLOWED_DOMAINS) -> bool:
    try:
        parsed = urlparse(url)
        if parsed.scheme not in {"http", "https"}:
            return False
        if parsed.netloc.lower() not in allowed_domains:
            return False
        if parsed.path.lower().endswith(BAD_FILE_EXTENSIONS):
            return False
        return True
    except Exception:
        return False

def extract_links(base_url: str, html: str):
    soup = BeautifulSoup(html, "html.parser")
    links = set()
    for a in soup.find_all("a", href=True):
        joined = normalize_url(urljoin(base_url, a["href"]))
        if is_allowed_url(joined):
            links.add(joined)
    return links

## Step 4. Crawl the Trinity website

For classroom runtime, start with `max_pages=150` or `300`, then scale up later.

In [10]:
def crawl_site(seed_urls, max_pages=200, delay=0.5, timeout=15):
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (compatible; ClassExerciseBot/1.0; +https://www.trincoll.edu/)"
    })

    visited = set()
    discovered = deque(seed_urls)
    pages = []

    with tqdm(total=max_pages, desc="Crawling") as pbar:
        while discovered and len(visited) < max_pages:
            url = discovered.popleft()
            if url in visited:
                continue

            try:
                resp = session.get(url, timeout=timeout)
                content_type = resp.headers.get("Content-Type", "")
                if "text/html" not in content_type:
                    visited.add(url)
                    continue

                html = resp.text
                visited.add(url)

                pages.append({
                    "url": url,
                    "status_code": resp.status_code,
                    "html": html
                })

                for link in extract_links(url, html):
                    if link not in visited:
                        discovered.append(link)

                pbar.update(1)
                time.sleep(delay)

            except Exception as e:
                visited.add(url)
                print(f"Failed: {url} -> {e}")

    return pd.DataFrame(pages)

## Step 5. Run the crawler

In [11]:
pages_df = crawl_site(SEED_URLS, max_pages=200, delay=0.25)
pages_df.head()

Crawling:   0%|          | 0/200 [00:00<?, ?it/s]

,url,status_code,html
0,https://www.trincoll.edu/,200,"<!DOCTYPE html>\n<html lang=""en-US"" class=""no-..."
1,https://bantamsports.com/,200,"\r\n\r\n<!doctype html>\r\n<html id=""ctl00_htm..."
2,https://www.trincoll.edu,200,"<!DOCTYPE html>\n<html lang=""en-US"" class=""no-..."
3,https://www.trincoll.edu/current-students,200,"<!DOCTYPE html>\n<html lang=""en-US"" class=""no-..."
4,https://www.trincoll.edu/sustainability,200,"<!DOCTYPE html>\n<html lang=""en-US"" class=""no-..."


## Step 6. Extract clean text from each page

In [12]:
def html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator=" ", strip=True)
    text = re.sub(r"\s+", " ", text).strip()
    return text

pages_df["text"] = pages_df["html"].apply(html_to_text)
pages_df["n_chars"] = pages_df["text"].str.len()

pages_df = pages_df[pages_df["n_chars"] > 200].copy()
pages_df = pages_df.drop_duplicates(subset="url").reset_index(drop=True)

pages_df[["url", "n_chars"]].head()

,url,n_chars
0,https://www.trincoll.edu/,5595
1,https://bantamsports.com/,3236
2,https://www.trincoll.edu,5595
3,https://www.trincoll.edu/current-students,1579
4,https://www.trincoll.edu/sustainability,2916


## Step 7. Inspect a few pages

In [13]:
for i in range(min(3, len(pages_df))):
    print("=" * 100)
    print("URL:", pages_df.loc[i, "url"])
    print()
    print(pages_df.loc[i, "text"][:1000], "...")
    print()

URL: https://www.trincoll.edu/

Welcome to Trinity College | Trinity College Trinity College Apply/Visit Request Info Give Menu Welcome to the Coop Explore Trinity ﻿ Academic Experience Benefit from world-class, well-rounded academics ﻿ Community Life Learn about campus happenings ﻿ Athletic Excellence Explore Bantam athletics ﻿ Possibility Starts Here Explore the Trinity experience. Previous Next Find your passion. Students have many opportunities to explore—more than 50 majors, 50 minors, 200+ opportunities for internships, research, and co-curricular experiences. Explore Your Options ﻿ Succeed after graduation. The Career and Life Design Center supports students (and alumni) after graduation. Ninety-five percent of students started their careers, graduate education, or postgrad experiences six months after graduation. Get to know the Career and Life Design Center ﻿ Benefit from our urban campus. Located in the dynamic capital city of Hartford, Trinity is close to many opportunities 

## Step 8. Load a pretrained embedding model

We will use the small and commonly used Sentence Transformer model `all-MiniLM-L6-v2`.

In [14]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Step 9. Example: generate an embedding for one document/page

In [15]:
sample_text = pages_df.loc[0, "text"]
sample_embedding = model.encode(sample_text, convert_to_numpy=True)

print("Embedding shape:", sample_embedding.shape)
print(sample_embedding[:10])

Embedding shape: (384,)
[ 0.03213356 -0.05316025  0.0060986  -0.01993905  0.01368547 -0.0073042
 -0.09048484 -0.04328989 -0.02437817  0.04900302]


## Step 10. Generate embeddings for all crawled pages

In [16]:
page_texts = pages_df["text"].tolist()

page_embeddings = model.encode(
    page_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings matrix shape:", page_embeddings.shape)

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Embeddings matrix shape: (200, 384)


## Step 11. Store embeddings with the page data

In [17]:
pages_df["embedding"] = list(page_embeddings)
pages_df[["url", "embedding"]].head()

,url,embedding
0,https://www.trincoll.edu/,"[0.032133553, -0.053160153, 0.0060985666, -0.0..."
1,https://bantamsports.com/,"[-0.040173225, -0.07661107, 0.057185423, -0.07..."
2,https://www.trincoll.edu,"[0.032133553, -0.053160153, 0.0060985666, -0.0..."
3,https://www.trincoll.edu/current-students,"[-0.017335387, -0.07971859, 0.0073806257, -0.0..."
4,https://www.trincoll.edu/sustainability,"[0.011829818, 0.0011930838, 0.112851076, 0.035..."


## Step 12. Cosine similarity between two documents

In [18]:
doc1 = pages_df.loc[0, "embedding"].reshape(1, -1)
doc2 = pages_df.loc[1, "embedding"].reshape(1, -1)

sim = cosine_similarity(doc1, doc2)[0, 0]
print("Cosine similarity between document 1 and document 2:", sim)

Cosine similarity between document 1 and document 2: 0.5466554


## Step 13. Build a search function for natural-language queries

In [19]:
def semantic_search(query, model, pages_df, top_k=5):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    page_matrix = np.vstack(pages_df["embedding"].values)
    sims = cosine_similarity(query_embedding, page_matrix)[0]

    result_df = pages_df[["url", "text"]].copy()
    result_df["score"] = sims
    result_df = result_df.sort_values("score", ascending=False).reset_index(drop=True)

    return result_df.head(top_k)

## Step 14. Example search

In [20]:
results = semantic_search("undergraduate admissions", model, pages_df, top_k=5)
results[["url", "score"]]

,url,score
0,https://www.trincoll.edu/abouttrinity,0.468648
1,https://www.trincoll.edu/academics,0.431011
2,https://www.trincoll.edu/academics/majors-and-...,0.425218
3,https://www.trincoll.edu/admissions,0.407292
4,https://www.trincoll.edu/people-directory,0.384982


## Step 15. Student Task: write your own query search code

Write code that:
1. takes a query string,
2. embeds the query,
3. computes cosine similarity with all crawled pages,
4. returns the page with the highest cosine similarity.

In [21]:
# TODO: Students complete this function

def find_best_page_for_query(query, model, pages_df):
    # 1. Generate query embedding
    # query_embedding = ...
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # 2. Stack page embeddings into a matrix
    # page_matrix = ...
    page_matrix = np.vstack(pages_df["embedding"].values)
    
    # 3. Compute cosine similarity
    # sims = ...
    sims = cosine_similarity(query_embedding, page_matrix)[0]

    # 4. Find the best page index
    # best_idx = ...
    best_idx = np.argmax(sims)
    # 5. Return URL, score, and maybe a text snippet
    # return ...
    return {
        "url": pages_df.loc[best_idx, "url"],
        "score": sims[best_idx],
        "text_snippet": pages_df.loc[best_idx, "text"][:500] + "..."
    }


## Step 16. Ground-truth queries for evaluation

In [22]:
ground_truth = {
    "admissions": [
        "https://www.trincoll.edu/admissions/undergraduate-admissions"
    ],
    "financial aid": [
        "https://www.trincoll.edu/admissions/finaid"
    ],
    "visit campus": [
        "https://www.trincoll.edu/admissions/visit-trinity"
    ],
    "academics": [
        "https://www.trincoll.edu/academics"
    ],
    "registrar": [
        "https://www.trincoll.edu/registrar"
    ],
    "student life": [
        "https://www.trincoll.edu/studentlife"
    ],
    "library": [
        "https://www.trincoll.edu/lits"
    ],
    "current students": [
        "https://www.trincoll.edu/current-students"
    ],
    "about trinity": [
        "https://www.trincoll.edu/abouttrinity"
    ],
    "sports": [
        "https://bantamsports.com"
    ],
}

## Step 17. Check which ground-truth pages are in your crawl

In [23]:
crawled_urls = set(pages_df["url"].tolist())

for query, gt_urls in ground_truth.items():
    found = [u for u in gt_urls if u in crawled_urls]
    print(f"{query:15s} -> found {len(found)} / {len(gt_urls)}")
    if not found:
        print("   Replace with the closest matching crawled URL.")

admissions      -> found 0 / 1
   Replace with the closest matching crawled URL.
financial aid   -> found 0 / 1
   Replace with the closest matching crawled URL.
visit campus    -> found 0 / 1
   Replace with the closest matching crawled URL.
academics       -> found 1 / 1
registrar       -> found 1 / 1
student life    -> found 1 / 1
library         -> found 0 / 1
   Replace with the closest matching crawled URL.
current students -> found 1 / 1
about trinity   -> found 1 / 1
sports          -> found 1 / 1


## Step 18. Search all 10 keywords and inspect the top results

In [24]:
search_outputs = {}

for query in ground_truth.keys():
    result_df = semantic_search(query, model, pages_df, top_k=5)
    search_outputs[query] = result_df
    print("=" * 100)
    print("QUERY:", query)
    print(result_df[["url", "score"]])

QUERY: admissions
                                           url     score
0          https://www.trincoll.edu/admissions  0.455520
1        https://www.trincoll.edu/abouttrinity  0.428719
2  https://www.trincoll.edu/registrar/students  0.403189
3           https://www.trincoll.edu/academics  0.398023
4            https://www.trincoll.edu/test-hhd  0.386045
QUERY: financial aid
                                         url     score
0        https://www.trincoll.edu/admissions  0.362965
1       https://www.trincoll.edu/trinityplus  0.358651
2  https://www.trincoll.edu/student-accounts  0.348566
3  https://www.trincoll.edu/Dean-Of-Students  0.338006
4            https://www.trincoll.edu/j-term  0.332784
QUERY: visit campus
                                           url     score
0        https://www.trincoll.edu/abouttrinity  0.536243
1  https://www.trincoll.edu/campus-safety/SART  0.520641
2    https://www.trincoll.edu/Dean-Of-Students  0.515565
3       https://www.trincoll.edu/campus-s

## Step 19. Evaluation metrics

We will report:
- Top-1 Accuracy
- Recall@5
- MRR

Use LLms to learn more about Top-1 Accuracy, Recall@5, MRR

In [25]:
def evaluate_search(ground_truth, search_outputs, k=5):
    total = len(ground_truth)
    top1_correct = 0
    recall_at_k = 0
    reciprocal_ranks = []

    for query, gt_urls in ground_truth.items():
        gt_set = {normalize_url(u) for u in gt_urls}
        preds = [normalize_url(u) for u in search_outputs[query]["url"].tolist()[:k]]

        if len(preds) > 0 and preds[0] in gt_set:
            top1_correct += 1

        found_rank = None
        for rank, pred in enumerate(preds, start=1):
            if pred in gt_set:
                found_rank = rank
                break

        if found_rank is not None:
            recall_at_k += 1
            reciprocal_ranks.append(1.0 / found_rank)
        else:
            reciprocal_ranks.append(0.0)

    return {
        "top1_accuracy": top1_correct / total,
        f"recall@{k}": recall_at_k / total,
        "mrr": sum(reciprocal_ranks) / total,
    }

## Step 20. Run the evaluation

In [26]:
metrics = evaluate_search(ground_truth, search_outputs, k=5)
metrics

{'top1_accuracy': 0.2, 'recall@5': 0.4, 'mrr': 0.25833333333333336}

## Step 21. Required class exercise submission

### Part A. Implementation
Show:
1. code to crawl Trinity College links,
2. code to extract text from each page,
3. code to generate embeddings with a pretrained model,
4. code to compute cosine similarity,
5. code to search with a natural-language query.

### Part B. Search experiments
Run the system on these 10 queries:
- admissions
- financial aid
- visit campus
- academics
- registrar
- student life
- library
- current students
- about trinity
- sports

### Part C. Evaluation
Use the provided ground truth or your corrected crawled version of it, then report:
- Top-1 Accuracy
- Recall@5
- MRR


## Submission guideline
After completion, upload the notebook by class time in the moodle(whatever you have). If you need more time, complete it by 04/11 and upload another version(name the file as 'version 2') in the moodle.  